In [ ]:
# Data preparation (within-Mexico)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from docx import Document
from docx.shared import Pt, Inches
from pathlib import Path

sns.set_theme(style="whitegrid")

def _find_root(markers=('data', 'WESEffectsDiD')):
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if all((cand / m).exists() for m in markers):
            return cand
    raise FileNotFoundError(
        'Could not locate the repo root (expected folders data/ and '
        f'WESEffectsDiD/). Searched upward from {here}.')

ROOT = _find_root()
RESULTS = ROOT / 'mexico' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

SEX_TO_FEMALE = {'Women+': 1, 'Women': 1, 'Female': 1,
                 'Men+': 0,  'Men': 0,   'Male': 0}

def recode_sex(s):
    """Map a raw Sex label series onto the {0 = male, 1 = female} indicator."""
    out = s.astype(str).str.strip().map(SEX_TO_FEMALE)
    if out.isna().any():
        bad = sorted(s[out.isna()].astype(str).str.strip().unique())
        raise ValueError(f"Unrecognised Sex label(s): {bad}")
    return out.astype(int)

df = pd.read_csv(ROOT / 'data' / 'mexico' / 'aggregatedData.csv').rename(columns={
    'State': 'state', 'Year': 'year',
    'SelfEmployedPSTS': 'psts_self_employed', 'LaborForce': 'labor_force'})
df['female'] = recode_sex(df['Sex'])
df['gender'] = df['female'].map({0: 'Male', 1: 'Female'})
df['rate']     = df['psts_self_employed'] / df['labor_force']
df['log_rate'] = np.log(df['rate'])

df_final = df[df['year'].between(2014, 2017)].copy()
df_final['year'] = df_final['year'].astype(int)

print('Pre-treatment rows 2014-2017:', len(df_final),
      '| states:', df_final['state'].nunique(),
      '| years:', sorted(df_final['year'].unique()))
df_final.head()

## 1. Within-Mexico parallel trends (male vs female)

In [ ]:
# Within-Mexico parallel trends + OLS + F-test (All PSTS)
df_pre = df_final.dropna(subset=['log_rate', 'labor_force']).copy()

df_plot = df.dropna(subset=['log_rate', 'labor_force']).copy()
df_plot['year'] = df_plot['year'].astype(int)

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_plot, x='year', y='log_rate', hue='gender', marker='o', linewidth=2.5)
plt.axvline(x=2018, color='red', linestyle='--', label='Treatment cutoff (2018)')
plt.title('Mexico Parallel Trends: All PSTS self-employed (log Rate)', fontsize=14)
plt.ylabel('Log(PSTS self-employed / Labor Force)')
plt.xticks(sorted(df_plot['year'].unique()))
plt.legend()
plt.savefig(RESULTS / 'mexico_parallelTrends_psts_all.png', dpi=300, bbox_inches='tight')
plt.show()

model = smf.ols("log_rate ~ female * C(year) + C(state)", data=df_pre)
res = model.fit(cov_type='cluster', cov_kwds={'groups': df_pre['state']})
print(res.summary())

interaction_vars = [v for v in res.params.index if 'female:C(year)' in v]
f_test = res.f_test(interaction_vars)
print('\n--- F-test on female:C(year) interactions (All PSTS) ---')
print(f_test.summary())

doc = Document()
sec = doc.sections[0]; sec.page_width = Inches(11); sec.page_height = Inches(8.5)
doc.add_heading('Mexico OLS Parallel Trends — All PSTS', 0)
r = doc.add_paragraph().add_run(str(res.summary())); r.font.name = 'Courier New'; r.font.size = Pt(9)
doc.save(str(RESULTS / "Mexico_ParallelTrends_OLS_allPSTS.docx"))

doc = Document()
doc.add_heading('Mexico F-Test — All PSTS', 0)
r = doc.add_paragraph().add_run(str(f_test.summary())); r.font.name = 'Courier New'; r.font.size = Pt(9)
doc.save(str(RESULTS / "Mexico_FTest_allPSTS.docx"))

## 2. Cross-country pre-trends: Canada vs Mexico (DDD)

In [ ]:
# ---- Shared data prep: pool Canada + Mexico (both genders) for the DDD ----
sns.set_theme(style="whitegrid")

ca_se = pd.read_csv(ROOT / 'WESEffectsDiD' / 'data' / 'cleanedSelfEmployed.csv')
ca_lf = pd.read_csv(ROOT / 'WESEffectsDiD' / 'data' / 'cleanedLabourStats.csv')
ca = pd.merge(ca_se,
              ca_lf[['Province', 'Year', 'Sex', 'Control_LaborForce']],
              on=['Province', 'Year', 'Sex'], how='left')
ca['female'] = recode_sex(ca['Sex'])
ca = ca.rename(columns={'Province': 'geo', 'Year': 'year',
                        'Self_Employed': 'num', 'Control_LaborForce': 'lf'})
ca['country'] = 'Canada'
ca['canada'] = 1
ca_base = ca[['country', 'canada', 'geo', 'year', 'female', 'num', 'lf']].copy()

mx = pd.read_csv(ROOT / 'data' / 'mexico' / 'aggregatedData.csv')

def _mexico_long():
    """Mexico (long) -> pooled schema for the all-PSTS self-employed numerator."""
    out = mx.rename(columns={'State': 'geo', 'Year': 'year',
                             'SelfEmployedPSTS': 'num', 'LaborForce': 'lf'}).copy()
    out['female'] = recode_sex(out['Sex'])
    out['country'] = 'Mexico'; out['canada'] = 0
    return out[['country', 'canada', 'geo', 'year', 'female', 'num', 'lf']]

def build_pooled():
    """Pooled Canada+Mexico long frame, pre-treatment 2014-2017, with log-rate outcome."""
    pooled = pd.concat([ca_base, _mexico_long()], ignore_index=True)
    pooled['geo'] = pooled['country'] + ':' + pooled['geo'].astype(str)
    pooled = pooled[pooled['year'].between(2014, 2017)].copy()
    pooled['year'] = pooled['year'].astype(int)
    pooled['rate'] = pooled['num'] / pooled['lf']
    pooled['log_rate'] = np.log(pooled['rate'])
    pooled = pooled.dropna(subset=['log_rate', 'lf'])
    pooled = pooled[np.isfinite(pooled['log_rate'])].copy()
    return pooled

DDD_FORMULA = ("log_rate ~ C(geo) + C(year) + female + female:C(year) "
               "+ canada:C(year) + female:canada + female:canada:C(year)")

def fit_ddd(pooled):
    res = smf.ols(DDD_FORMULA, data=pooled).fit(
        cov_type='cluster', cov_kwds={'groups': pooled['geo']})
    triple = [v for v in res.params.index
              if ('female' in v) and ('canada' in v) and ('C(year)' in v)]
    return res, res.f_test(triple), triple

def ddd_plot(pooled, title, outfile):
    """Left: mean log-rate by country x gender. Right: gender gap (women-men) per country."""
    p = pooled.copy()
    p['Gender'] = p['female'].map({0: 'Men', 1: 'Women'})
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    sns.lineplot(ax=axes[0], data=p, x='year', y='log_rate', hue='country',
                 style='Gender', markers=True, dashes=True, linewidth=2.2, errorbar=None)
    axes[0].set_title('Mean log(PSTS self-employed / labour force)')
    axes[0].set_ylabel('Log rate'); axes[0].set_xticks(sorted(p['year'].unique()))
    gap = (p.groupby(['country', 'year', 'female'])['log_rate'].mean().unstack('female'))
    gap['gap'] = gap[1] - gap[0]
    gap = gap.reset_index()
    sns.lineplot(ax=axes[1], data=gap, x='year', y='gap', hue='country',
                 marker='o', linewidth=2.6)
    axes[1].set_title('Gender gap = log(women) - log(men)  [DDD object]')
    axes[1].set_ylabel('Female - Male (log points)'); axes[1].set_xticks(sorted(p['year'].unique()))
    fig.suptitle(title, fontsize=14); fig.tight_layout()
    fig.savefig(outfile, dpi=300, bbox_inches='tight'); plt.show()

print('Canada rows:', len(ca_base), '| Mexico states:', mx['State'].nunique())

In [ ]:
# Cross-country DDD (All PSTS): Mexico SelfEmployedPSTS vs Canada self-employed
pooled = build_pooled()
print('Pooled rows:', len(pooled),
      '| clusters (geo):', pooled['geo'].nunique(),
      '| Canada:', int((pooled.country == 'Canada').sum()),
      '| Mexico:', int((pooled.country == 'Mexico').sum()),
      '| years:', sorted(pooled['year'].unique()))

ddd_plot(pooled,
         'Canada vs Mexico - Cross-Country Pre-Trends (All PSTS), 2014-2017',
         RESULTS / 'canadaMexico_ddd_parallelTrends_allPSTS.png')

res_ddd, ftest_ddd, triple_ddd = fit_ddd(pooled)
print(res_ddd.summary())
print('\n--- DDD F-test on female:canada:C(year) interactions (All PSTS) ---')
print('Tested terms:', triple_ddd)
print(ftest_ddd.summary())

doc = Document()
sec = doc.sections[0]; sec.page_width = Inches(11); sec.page_height = Inches(8.5)
doc.add_heading('Canada vs Mexico - DDD Pre-Trends OLS - All PSTS', 0)
r = doc.add_paragraph().add_run(str(res_ddd.summary())); r.font.name = 'Courier New'; r.font.size = Pt(9)
doc.save(str(RESULTS / 'CanadaMexico_DDD_OLS_allPSTS.docx'))

doc = Document()
doc.add_heading('Canada vs Mexico - DDD F-Test (female:canada:year) - All PSTS', 0)
r = doc.add_paragraph().add_run(str(ftest_ddd.summary())); r.font.name = 'Courier New'; r.font.size = Pt(9)
doc.save(str(RESULTS / 'CanadaMexico_DDD_FTest_allPSTS.docx'))

## 3. Control DiD (Mexico) — common-shock check

In [ ]:
# Control DiD (Mexico) — same specification as the Canadian DiD
did = df.dropna(subset=['log_rate', 'labor_force', 'UnemploymentRate']).copy()
did = did[np.isfinite(did['log_rate'])].copy()
did['year']  = did['year'].astype(int)
did['Treat'] = did['female']
did['Post']  = (did['year'] >= 2018).astype(int)

did_formula = ('log_rate ~ Treat + Treat:Post + UnemploymentRate '
               '+ C(state) + C(year)')
res_did = smf.ols(did_formula, data=did).fit(
    cov_type='cluster', cov_kwds={'groups': did['state']})
print(res_did.summary())
print('\n--- Mexico control DiD: treatment effect (Treat:Post) ---')
print('coef =', round(res_did.params['Treat:Post'], 4),
      '| std err =', round(res_did.bse['Treat:Post'], 4),
      '| p =', round(res_did.pvalues['Treat:Post'], 4),
      '| n =', int(res_did.nobs), '| states:', did['state'].nunique())

doc = Document()
sec = doc.sections[0]; sec.page_width = Inches(11); sec.page_height = Inches(8.5)
doc.add_heading('Mexico Control DiD — All PSTS (Canadian specification)', 0)
r = doc.add_paragraph().add_run(str(res_did.summary())); r.font.name = 'Courier New'; r.font.size = Pt(9)
doc.save(str(RESULTS / 'Mexico_ControlDiD_OLS_allPSTS.docx'))